# 성능튜닝

## 1.환경준비

### (1) import

In [1]:
#라이브러리들을 불러오자.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# 전처리
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler

# 모델링
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.metrics import * 

import warnings    # 경고메시지 제외
warnings.filterwarnings(action='ignore')

### (2) 데이터 준비

* 변수설명
    * COLLEGE : 대학 졸업여부
    * INCOME : 연수입
    * OVERAGE : 월평균 초과사용 시간(분)
    * LEFTOVER : 월평균 잔여시간비율(%)
    * HOUSE : 집값
    * HANDSET_PRICE : 스마트폰 가격
    * OVER_15MINS_CALLS_PER_MONTH : 월평균 장기통화(15분이상) 횟수
    * AVERAGE_CALL_DURATION : 평균 통화 시간
    * REPORTED_SATISFACTION : 만족도 설문조사 결과
    * REPORTED_USAGE_LEVEL : 사용도 자가진단 결과
    * CONSIDERING_CHANGE_OF_PLAN : 향후 변경계획 설문조사 결과
    * CHURN : 이탈(번호이동) 여부 (1-이탈, 0-잔류, Target 변수)


In [2]:
# 데이터를 불러옵시다.
path = 'https://raw.githubusercontent.com/DA4BAM/dataset/master/mobile_cust_churn.csv'
data = pd.read_csv(path)
data = data.sample(5000, random_state = 2022)
data['CHURN'] = data['CHURN'].map({'LEAVE':1, 'STAY':0}) #sklearn에서는 y를 가변수화 할 필요가 없다. -> 반드시필요는 statsmodels, tensorflow(keras)
data.head()

,id,COLLEGE,INCOME,OVERAGE,LEFTOVER,HOUSE,HANDSET_PRICE,OVER_15MINS_CALLS_PER_MONTH,AVERAGE_CALL_DURATION,REPORTED_SATISFACTION,REPORTED_USAGE_LEVEL,CONSIDERING_CHANGE_OF_PLAN,CHURN
3178,3179,0,119512,51,31,248566,229,5,2,very_sat,very_high,considering,1
14926,14927,1,142144,192,15,774317,581,29,4,unsat,very_little,never_thought,1
15116,15117,1,142308,0,79,306426,497,1,1,sat,little,considering,0
12733,12734,1,113385,0,0,333599,819,1,6,very_unsat,very_high,considering,1
14032,14033,1,90348,209,10,637286,360,26,4,unsat,little,actively_looking_into_it,0


## 2.데이터 준비

### (1) 데이터 정리

In [3]:
drop_cols = ['id']
data.drop(drop_cols, axis = 1, inplace = True )

### (2) 데이터분할1 : x, y 나누기

In [4]:
target = 'CHURN'
x = data.drop(target, axis = 1)
y = data.loc[:, target]

### (3) NA 조치

### (4) 가변수화

In [5]:
dumm_cols = ['REPORTED_SATISFACTION','REPORTED_USAGE_LEVEL','CONSIDERING_CHANGE_OF_PLAN']
x = pd.get_dummies(x, columns = dumm_cols, drop_first = True)

### (5) 데이터분할2 : train : validation 나누기

In [6]:
x_train, x_val, y_train, y_val = train_test_split(x, y, test_size = .3, random_state = 20)

### (6) Scaling

In [7]:
scaler = MinMaxScaler()
x_train_s = scaler.fit_transform(x_train)
x_val_s = scaler.transform(x_val)

## 3.선형모델 튜닝

Logistic Regression : 전진선택법
* 변수를 하나씩 늘려가면서
* AIC를 가장 낮추는 모델 찾기

### (1) 전진선택을 수행할 함수 만들기( **로지스틱 회귀** 용)

In [8]:
# 아래 함수는 로지스틱 회귀를 위한 전진선택법 함수 입니다.
import statsmodels.api as sm

def forward_stepwise_logistic(x_train, y_train):

    # 변수목록, 선택된 변수 목록, 단계별 모델과 AIC 저장소 정의
    features = list(x_train)
    selected = []
    step_df = pd.DataFrame({ 'step':[], 'feature':[],'aic':[]})

    # 
    for s in range(0, len(features)) :
        result =  { 'step':[], 'feature':[],'aic':[]}

        # 변수 목록에서 변수 한개씩 뽑아서 모델에 추가
        for f in features :
            vars = selected + [f]
            x_tr = x_train[vars]
            model = sm.Logit(y_train, x_tr).fit()
            result['step'].append(s+1)
            result['feature'].append(vars)
            result['aic'].append(model.aic)
        
        # 모델별 aic 집계
        temp = pd.DataFrame(result).sort_values('aic').reset_index(drop = True)

        # 만약 이전 aic보다 새로운 aic 가 크다면 멈추기
        if step_df['aic'].min() < temp['aic'].min() :
            break
        step_df = pd.concat([step_df, temp], axis = 0).reset_index(drop = True)

        # 선택된 변수 제거
        v = temp.loc[0,'feature'][s]
        features.remove(v)

        selected.append(v)
    
    # 선택된 변수와 step_df 결과 반환
    return selected, step_df

### (2) 전진선택법 수행

In [9]:
vars, result = forward_stepwise_logistic(x_train, y_train)

Optimization terminated successfully.
         Current function value: 0.693075
         Iterations 3
Optimization terminated successfully.
         Current function value: 0.693062
         Iterations 2
Optimization terminated successfully.
         Current function value: 0.683528
         Iterations 4
Optimization terminated successfully.
         Current function value: 0.693007
         Iterations 3
Optimization terminated successfully.
         Current function value: 0.684909
         Iterations 2
Optimization terminated successfully.
         Current function value: 0.693042
         Iterations 2
Optimization terminated successfully.
         Current function value: 0.686899
         Iterations 4
Optimization terminated successfully.
         Current function value: 0.692565
         Iterations 3
Optimization terminated successfully.
         Current function value: 0.692806
         Iterations 4
Optimization terminated successfully.
         Current function value: 0.693137
  

* 선택된 변수

In [10]:
vars

['OVERAGE',
 'HOUSE',
 'HANDSET_PRICE',
 'LEFTOVER',
 'REPORTED_SATISFACTION_very_sat',
 'INCOME',
 'REPORTED_SATISFACTION_sat']

In [11]:
list(x_train)

['COLLEGE',
 'INCOME',
 'OVERAGE',
 'LEFTOVER',
 'HOUSE',
 'HANDSET_PRICE',
 'OVER_15MINS_CALLS_PER_MONTH',
 'AVERAGE_CALL_DURATION',
 'REPORTED_SATISFACTION_sat',
 'REPORTED_SATISFACTION_unsat',
 'REPORTED_SATISFACTION_very_sat',
 'REPORTED_SATISFACTION_very_unsat',
 'REPORTED_USAGE_LEVEL_high',
 'REPORTED_USAGE_LEVEL_little',
 'REPORTED_USAGE_LEVEL_very_high',
 'REPORTED_USAGE_LEVEL_very_little',
 'CONSIDERING_CHANGE_OF_PLAN_considering',
 'CONSIDERING_CHANGE_OF_PLAN_never_thought',
 'CONSIDERING_CHANGE_OF_PLAN_no',
 'CONSIDERING_CHANGE_OF_PLAN_perhaps']

In [12]:
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', -1)

In [13]:
result

,step,feature,aic
0,1.0,[OVERAGE],4786.699456
1,1.0,[HOUSE],4796.363859
2,1.0,[OVER_15MINS_CALLS_PER_MONTH],4810.294604
3,1.0,[REPORTED_SATISFACTION_very_sat],4845.064834
4,1.0,[AVERAGE_CALL_DURATION],4849.951663
...,...,...,...
114,7.0,"[OVERAGE, HOUSE, HANDSET_PRICE, LEFTOVER, REPORTED_SATISFACTION_very_sat, INCOME, REPORTED_SATISFACTION_very_unsat]",4442.309949
115,7.0,"[OVERAGE, HOUSE, HANDSET_PRICE, LEFTOVER, REPORTED_SATISFACTION_very_sat, INCOME, CONSIDERING_CHANGE_OF_PLAN_never_thought]",4442.315315
116,7.0,"[OVERAGE, HOUSE, HANDSET_PRICE, LEFTOVER, REPORTED_SATISFACTION_very_sat, INCOME, REPORTED_USAGE_LEVEL_high]",4442.316816
117,7.0,"[OVERAGE, HOUSE, HANDSET_PRICE, LEFTOVER, REPORTED_SATISFACTION_very_sat, INCOME, REPORTED_USAGE_LEVEL_very_little]",4442.448187


### (3) 모델링

* 전체 변수 

In [14]:
m1 = LogisticRegression()
m1.fit(x_train, y_train)
p1 = m1.predict(x_val)

print(accuracy_score(y_val, p1))
print(classification_report(y_val, p1))

0.6333333333333333
              precision    recall  f1-score   support

           0       0.62      0.68      0.65       738
           1       0.65      0.59      0.62       762

    accuracy                           0.63      1500
   macro avg       0.63      0.63      0.63      1500
weighted avg       0.64      0.63      0.63      1500



* 전진선택법 변수

In [15]:
m2 = LogisticRegression()
m2.fit(x_train[vars], y_train)
p2 = m2.predict(x_val[vars])

print(accuracy_score(y_val, p2))
print(classification_report(y_val, p2))

0.634
              precision    recall  f1-score   support

           0       0.62      0.68      0.65       738
           1       0.66      0.59      0.62       762

    accuracy                           0.63      1500
   macro avg       0.64      0.63      0.63      1500
weighted avg       0.64      0.63      0.63      1500



## 4.하이퍼파라미터 튜닝

### (1) 필요한 함수 불러오기 

In [16]:
from sklearn.model_selection import RandomizedSearchCV, GridSearchCV

### (2) Random Search

① 값의 범위를 지정한다.  
② 모델 선언(시도 횟수 지정)  
③ 모델링(값의 범위 내에서 시도 횟수만큼 랜덤하게 선택해서 시도한다.)  
④ 가장 성능이 좋은 값을 선정


#### ① 값의 범위를 지정한다.

In [17]:
# dictionary형태로 선언
params = { 'n_neighbors' : range(1,51), 'metric' : ['euclidean', 'manhattan']  }
params

{'n_neighbors': range(1, 51), 'metric': ['euclidean', 'manhattan']}

#### ② 모델 선언

In [18]:
# 기본모델
model = KNeighborsClassifier()

# Random Search 설정.
model_rs = RandomizedSearchCV(model
                            , params              # hyperparameter 범위 지정.
                            , cv=5                    # k-fold Cross Validation
                            , n_iter=5                # Random하게 시도할 횟수
                            )

#### ③ 모델링

In [19]:
# 학습 : model이 아니라 model_rs
model_rs.fit(x_train_s, y_train)

RandomizedSearchCV(cv=5, estimator=KNeighborsClassifier(), n_iter=5,
                   param_distributions={'metric': ['euclidean', 'manhattan'],
                                        'n_neighbors': range(1, 51)})

In [20]:
# 튜닝 결과
model_rs.cv_results_

{'mean_fit_time': array([0.00260248, 0.0027595 , 0.00299492, 0.00649281, 0.00806055]),
 'std_fit_time': array([0.00047937, 0.00121493, 0.00199276, 0.00460191, 0.00252892]),
 'mean_score_time': array([0.15715408, 0.17115321, 0.17972198, 0.48015079, 0.44441323]),
 'std_score_time': array([0.01745585, 0.02596661, 0.02550769, 0.178347  , 0.10051914]),
 'param_n_neighbors': masked_array(data=[41, 47, 33, 49, 36],
              mask=[False, False, False, False, False],
        fill_value='?',
             dtype=object),
 'param_metric': masked_array(data=['euclidean', 'manhattan', 'manhattan', 'manhattan',
                    'euclidean'],
              mask=[False, False, False, False, False],
        fill_value='?',
             dtype=object),
 'params': [{'n_neighbors': 41, 'metric': 'euclidean'},
  {'n_neighbors': 47, 'metric': 'manhattan'},
  {'n_neighbors': 33, 'metric': 'manhattan'},
  {'n_neighbors': 49, 'metric': 'manhattan'},
  {'n_neighbors': 36, 'metric': 'euclidean'}],
 'split0_

In [21]:
model_rs.cv_results_['params']

[{'n_neighbors': 41, 'metric': 'euclidean'},
 {'n_neighbors': 47, 'metric': 'manhattan'},
 {'n_neighbors': 33, 'metric': 'manhattan'},
 {'n_neighbors': 49, 'metric': 'manhattan'},
 {'n_neighbors': 36, 'metric': 'euclidean'}]

In [22]:
model_rs.cv_results_['mean_test_score']

array([0.59657143, 0.61657143, 0.62142857, 0.622     , 0.59457143])

In [23]:
# 최적의 파라미터
model_rs.best_params_

{'n_neighbors': 49, 'metric': 'manhattan'}

In [24]:
# 그때의 성능
model_rs.best_score_

0.6220000000000001

In [25]:
# best 모델로 예측 및 평가
pred = model_rs.predict(x_val_s) #5개의 모델이 있는데 지가 알아서 좋은 놈으로 시도함
print(classification_report(y_val, pred))

              precision    recall  f1-score   support

           0       0.59      0.75      0.66       738
           1       0.67      0.49      0.57       762

    accuracy                           0.62      1500
   macro avg       0.63      0.62      0.62      1500
weighted avg       0.63      0.62      0.62      1500



### (3) 실습 : Random Search

* decision tree로 튜닝을 시도해 봅시다.
    * max_depth : 1~10
    * min_samples_leaf : 10 ~ 100

#### ① 값의 범위를 지정한다.

In [26]:
param={'max_depth':[1,2,3,4,5,6,7,8,9,10],
      'min_samples_leaf':list(range(10,101))}

#### ② 모델 선언

In [27]:
model=DecisionTreeClassifier()

#### ③ 모델링

In [28]:
model_rs=RandomizedSearchCV(model,param,cv=3,scoring='accuracy',n_iter=20)

In [29]:
model_rs.fit(x_train_s,y_train)

RandomizedSearchCV(cv=3, estimator=DecisionTreeClassifier(), n_iter=20,
                   param_distributions={'max_depth': [1, 2, 3, 4, 5, 6, 7, 8, 9,
                                                      10],
                                        'min_samples_leaf': [10, 11, 12, 13, 14,
                                                             15, 16, 17, 18, 19,
                                                             20, 21, 22, 23, 24,
                                                             25, 26, 27, 28, 29,
                                                             30, 31, 32, 33, 34,
                                                             35, 36, 37, 38, 39, ...]},
                   scoring='accuracy')

In [30]:
model_rs.cv_results_

{'mean_fit_time': array([0.02753107, 0.01965626, 0.01782513, 0.01659942, 0.01489814,
        0.00807722, 0.01374698, 0.02088873, 0.00732652, 0.01399477,
        0.02025342, 0.01294804, 0.01389742, 0.00802135, 0.01319345,
        0.01513664, 0.01109902, 0.02043732, 0.01238688, 0.01423446]),
 'std_fit_time': array([0.00621543, 0.00035428, 0.00412216, 0.00045495, 0.00052075,
        0.00078817, 0.00057099, 0.00176254, 0.00046141, 0.00160303,
        0.00381622, 0.00112378, 0.0007912 , 0.00084085, 0.00109816,
        0.00103876, 0.00125736, 0.0010367 , 0.00133522, 0.00307661]),
 'mean_score_time': array([0.00340438, 0.00197363, 0.00181055, 0.00135541, 0.00193055,
        0.00198698, 0.00199437, 0.0013303 , 0.00202998, 0.00166162,
        0.00261362, 0.00099746, 0.00117834, 0.00144474, 0.00248528,
        0.00166106, 0.00292126, 0.00151674, 0.00100358, 0.00228588]),
 'std_score_time': array([1.14614124e-03, 2.91696793e-05, 7.27106961e-04, 4.83337128e-04,
        1.44958278e-04, 7.64880121e-

In [31]:
model_rs.best_score_

0.6934296645457337

In [32]:
model_rs.best_params_

{'min_samples_leaf': 60, 'max_depth': 5}

In [33]:
pred=model_rs.predict(x_val)
print(classification_report(y_val,pred))

              precision    recall  f1-score   support

           0       0.62      0.42      0.50       738
           1       0.57      0.74      0.65       762

    accuracy                           0.59      1500
   macro avg       0.59      0.58      0.57      1500
weighted avg       0.59      0.59      0.58      1500



### (4) Grid Search

① 값의 범위를 지정한다.  
② 모델링(값의 범위 내에서 모든 조합을 다 시도한다.)  
③ 가장 성능이 좋은 값을 선정


#### ① 값의 범위를 지정한다.

In [34]:
# dictionary형태로 선언
params = { 'n_neighbors' : range(3,31,2), 'metric' : ['euclidean', 'manhattan']  }
params

{'n_neighbors': range(3, 31, 2), 'metric': ['euclidean', 'manhattan']}

#### ② 모델 선언

In [35]:
# 기본모델
model = KNeighborsClassifier()

# Random Search 설정.
model_gs = GridSearchCV(model, params, cv=5)

#### ③ 모델링

In [36]:
# 학습 : model이 아니라 model_rs
model_gs.fit(x_train_s, y_train)

GridSearchCV(cv=5, estimator=KNeighborsClassifier(),
             param_grid={'metric': ['euclidean', 'manhattan'],
                         'n_neighbors': range(3, 31, 2)})

In [37]:
# 튜닝 결과
model_gs.cv_results_

{'mean_fit_time': array([0.00335116, 0.00291142, 0.00244303, 0.00189137, 0.00262942,
        0.00212054, 0.00210772, 0.00178299, 0.00225954, 0.00259943,
        0.00309772, 0.00236588, 0.00188584, 0.00192313, 0.00157876,
        0.00145469, 0.00157514, 0.00172806, 0.0019351 , 0.00163698,
        0.00144429, 0.00158076, 0.00133867, 0.00176053, 0.00183773,
        0.00323682, 0.00175862, 0.00220523]),
 'std_fit_time': array([0.00186278, 0.00139047, 0.00045758, 0.0004861 , 0.0007796 ,
        0.00047269, 0.00067528, 0.00039326, 0.00036159, 0.00173766,
        0.00244186, 0.00044835, 0.00017738, 0.00062746, 0.00049802,
        0.00058582, 0.0005171 , 0.00038775, 0.00015333, 0.00044296,
        0.00053383, 0.00045266, 0.00053819, 0.00073925, 0.00037148,
        0.002563  , 0.00037753, 0.00039584]),
 'mean_score_time': array([0.14843316, 0.14869585, 0.10810452, 0.11704149, 0.11739554,
        0.10364079, 0.11810646, 0.10541415, 0.10274539, 0.10955248,
        0.10827031, 0.11307063, 0.108062

In [38]:
model_gs.cv_results_['params']

[{'metric': 'euclidean', 'n_neighbors': 3},
 {'metric': 'euclidean', 'n_neighbors': 5},
 {'metric': 'euclidean', 'n_neighbors': 7},
 {'metric': 'euclidean', 'n_neighbors': 9},
 {'metric': 'euclidean', 'n_neighbors': 11},
 {'metric': 'euclidean', 'n_neighbors': 13},
 {'metric': 'euclidean', 'n_neighbors': 15},
 {'metric': 'euclidean', 'n_neighbors': 17},
 {'metric': 'euclidean', 'n_neighbors': 19},
 {'metric': 'euclidean', 'n_neighbors': 21},
 {'metric': 'euclidean', 'n_neighbors': 23},
 {'metric': 'euclidean', 'n_neighbors': 25},
 {'metric': 'euclidean', 'n_neighbors': 27},
 {'metric': 'euclidean', 'n_neighbors': 29},
 {'metric': 'manhattan', 'n_neighbors': 3},
 {'metric': 'manhattan', 'n_neighbors': 5},
 {'metric': 'manhattan', 'n_neighbors': 7},
 {'metric': 'manhattan', 'n_neighbors': 9},
 {'metric': 'manhattan', 'n_neighbors': 11},
 {'metric': 'manhattan', 'n_neighbors': 13},
 {'metric': 'manhattan', 'n_neighbors': 15},
 {'metric': 'manhattan', 'n_neighbors': 17},
 {'metric': 'manha

In [39]:
model_gs.cv_results_['mean_test_score']

array([0.57885714, 0.57942857, 0.57714286, 0.57342857, 0.568     ,
       0.57457143, 0.58085714, 0.58257143, 0.57914286, 0.58057143,
       0.586     , 0.58457143, 0.57971429, 0.58028571, 0.57314286,
       0.59142857, 0.59171429, 0.59142857, 0.59828571, 0.59771429,
       0.59942857, 0.60142857, 0.60171429, 0.61057143, 0.61      ,
       0.61142857, 0.61857143, 0.618     ])

In [40]:
# 최적의 파라미터
model_gs.best_params_

{'metric': 'manhattan', 'n_neighbors': 27}

In [41]:
# 그때의 성능
model_gs.best_score_

0.6185714285714285

In [42]:
# best 모델로 예측 및 평가
pred = model_gs.predict(x_val_s)
print(classification_report(y_val, pred))

              precision    recall  f1-score   support

           0       0.59      0.72      0.65       738
           1       0.65      0.51      0.57       762

    accuracy                           0.61      1500
   macro avg       0.62      0.61      0.61      1500
weighted avg       0.62      0.61      0.61      1500



### (5) 실습 : Grid Search

* decision tree로 튜닝을 시도해 봅시다.
    * max_depth : 1~10
    * min_samples_leaf : 10 ~ 100

#### ① 값의 범위를 지정한다.

In [54]:
parma={'max_depth':range(1,11),'min_samples_leaf':range(10,101,10)}

#### ② 모델 선언

In [55]:
model2=DecisionTreeClassifier()

#### ③ 모델링

In [63]:
model2_gs=GridSearchCV(model2,parma,cv=5,verbose=5)

In [64]:
model2_gs.fit(x_train_s,y_train)

Fitting 5 folds for each of 100 candidates, totalling 500 fits
[CV 1/5] END ...............max_depth=1, min_samples_leaf=10; total time=   0.0s
[CV 2/5] END ...............max_depth=1, min_samples_leaf=10; total time=   0.0s
[CV 3/5] END ...............max_depth=1, min_samples_leaf=10; total time=   0.0s
[CV 4/5] END ...............max_depth=1, min_samples_leaf=10; total time=   0.0s
[CV 5/5] END ...............max_depth=1, min_samples_leaf=10; total time=   0.0s
[CV 1/5] END ...............max_depth=1, min_samples_leaf=20; total time=   0.0s
[CV 2/5] END ...............max_depth=1, min_samples_leaf=20; total time=   0.0s
[CV 3/5] END ...............max_depth=1, min_samples_leaf=20; total time=   0.0s
[CV 4/5] END ...............max_depth=1, min_samples_leaf=20; total time=   0.0s
[CV 5/5] END ...............max_depth=1, min_samples_leaf=20; total time=   0.0s
[CV 1/5] END ...............max_depth=1, min_samples_leaf=30; total time=   0.0s
[CV 2/5] END ...............max_depth=1, min_s

GridSearchCV(cv=5, estimator=DecisionTreeClassifier(),
             param_grid={'max_depth': range(1, 11),
                         'min_samples_leaf': range(10, 101, 10)},
             verbose=5)

In [65]:
model2_gs.cv_results_

{'mean_fit_time': array([0.01199098, 0.01395302, 0.01293631, 0.01042147, 0.01362305,
        0.00963907, 0.00870953, 0.00855322, 0.01003447, 0.00887928,
        0.01567307, 0.01169252, 0.0150485 , 0.01210566, 0.01205297,
        0.01523795, 0.0103076 , 0.01591311, 0.00968695, 0.00898824,
        0.01677232, 0.0142693 , 0.01364293, 0.0140914 , 0.01391082,
        0.01752815, 0.01403165, 0.01509247, 0.01584258, 0.01847339,
        0.01987033, 0.02903256, 0.01768217, 0.0233706 , 0.03914948,
        0.02700701, 0.0437428 , 0.04783149, 0.01891522, 0.02283387,
        0.02509518, 0.02340007, 0.02056694, 0.02574291, 0.02317371,
        0.0188221 , 0.02146764, 0.03147368, 0.02217069, 0.02147441,
        0.03522668, 0.02633219, 0.02423506, 0.02010698, 0.02138767,
        0.02071633, 0.02397537, 0.02623925, 0.01879382, 0.01801548,
        0.02182188, 0.02206264, 0.01738105, 0.0206604 , 0.01885324,
        0.01620836, 0.01476226, 0.01862936, 0.01504807, 0.01448421,
        0.02538743, 0.02109795,

In [66]:
model2_gs.best_params_

{'max_depth': 7, 'min_samples_leaf': 30}

In [67]:
model2_gs.best_score_ #하이퍼 파라미터 값을 조정 -> 모델 -> 검증
#튜닝해서 가장 좋은 결과

0.7005714285714285

In [68]:
pred=model2_gs.predict(x_val)

In [70]:
print(confusion_matrix(y_val,pred))
print('-'*50)
print(classification_report(y_val,pred))

[[313 425]
 [195 567]]
--------------------------------------------------
              precision    recall  f1-score   support

           0       0.62      0.42      0.50       738
           1       0.57      0.74      0.65       762

    accuracy                           0.59      1500
   macro avg       0.59      0.58      0.57      1500
weighted avg       0.59      0.59      0.58      1500

